<a href="https://colab.research.google.com/github/olorunfemibabalola/Computer-Vision-Learning/blob/main/08_Better_CV_Kaggle_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BETTER KAGGLE SUBMISSION

## Task 1: Download the dataset

You can access the competition data and submit your solutions via the [Kaggle website](https://www.kaggle.com/c/budl25) or using the [Kaggle API](https://github.com/Kaggle/kaggle-api). In the latter case all the interactions with Kaggle can be performed without leaving the notebook environment, so this is what we're going to use.

In [ ]:
# put Kaggle API credentials where they belong
!mkdir -p /root/.config/kaggle
!echo '{"username":"YOUR_USERNAME_GOES_HERE","key":"YOUR_KEY_GOES_HERE"}' > /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

In the cell below you will download the competition data and unzip it:

In [ ]:
import kaggle
!kaggle competitions download -c bucv26 --force
!unzip -o *.zip

100% 36.8M/36.8M [00:00<00:00, 120MB/s] 

Archive:  bucv26.zip
  inflating: sample_submission.csv   
  inflating: test_images/1144_7090.jpg  
  inflating: test_images/1148_7947.jpg  
  inflating: test_images/1290_2619.jpg  
  inflating: test_images/1301_4384.jpg  
  inflating: test_images/1334_1570.jpg  
  inflating: test_images/1455_3146.jpg  
  inflating: test_images/1513_1725.jpg  
  inflating: test_images/1590_4958.jpg  
  inflating: test_images/1597_3761.jpg  
  inflating: test_images/1651_2690.jpg  
  inflating: test_images/1733_5951.jpg  
  inflating: test_images/1750_4226.jpg  
  inflating: test_images/1758_4712.jpg  
  inflating: test_images/1998_7549.jpg  
  inflating: test_images/1999_9904.jpg  
  inflating: test_images/2049_8804.jpg  
  inflating: test_images/2084_9094.jpg  
  inflating: test_images/2092_7682.jpg  
  inflating: test_images/2143_5607.jpg  
  inflating: test_images/2169_7528.jpg  
  inflating: test_images/2171_1150.jpg  
  inflating: test_images/2193_5298.jpg

In [ ]:
!ls /content/

bucv26.zip   sample_submission.csv  train_images
sample_data  test_images	    train_labels.csv


Let's load the .csv file and see what it looks like:

In [ ]:
import pandas as pd
df = pd.read_csv('train_labels.csv')
df.head(5)

,anon_id,sex,bmi,filename
0,5956,0,21.7,5956_8270.jpg
1,5956,0,21.7,5956_1860.jpg
2,5956,0,21.7,5956_6390.jpg
3,5956,0,21.7,5956_6191.jpg
4,5956,0,21.7,5956_6734.jpg


In [ ]:
train_dir = '/content/train_images'
test_dir = '/content/test_images'

## Task 2: Prepare data for training

We afre not downsizing the images anymore; higher resolution often allows to build a better model.

Here we create a fairly standard and minimalistic custom Pytorch dataset.

In [ ]:
import os
from tqdm.auto import tqdm
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.transforms.v2 import Normalize, Compose, ToImage, ToDtype, Resize

class XrayDataset(Dataset):
    def __init__(self, img_dir, df_labels=None,transform=None):
        self.img_dir = img_dir
        self.df_labels = df_labels
        self.transform = transform

        self.img_files = os.listdir(img_dir)
        if df_labels is not None:
            self.file2label = dict(zip(df_labels['filename'], zip(df_labels['sex'], df_labels['bmi'])))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        image = Image.open(f'{self.img_dir}/{img_name}').convert('L') # our images are greyscale
        if self.transform: image = self.transform(image)
        label = self.file2label[img_name] if self.df_labels is not None else [-1, -1]
        return image, torch.tensor(label[0], dtype=torch.float32), torch.tensor(label[1], dtype=torch.float32)

In [ ]:
# You'll need to calculate the mean and std deviation of your dataset for proper normalization!
transforms = Compose([
    ToImage(),                           # Convert a tensor, ndarray, or PIL Image to torchvision.tv_tensors.Image
    Resize(224),                         # Images should all be 224x224 already but since it's not guranteed, we resize here
    ToDtype(torch.float32, scale=True),  # Converts the input to a specific dtype, optionally scaling the values to 0..1 range
    Normalize(mean=[0.5], std=[0.5])     # Normalization, currently using placeholder values!
])

# transforms is where you could add data augmentations too (like random flip or rotation) but for training data only!
ds = XrayDataset('/content/train_images', df, transform=transforms)
ds[0]

(Image([[[-1.0000, -1.0000, -1.0000,  ..., -0.7725, -0.7725, -0.7725],
         [-1.0000, -1.0000, -1.0000,  ..., -0.9216, -0.9216, -0.9216],
         [-1.0000, -1.0000, -1.0000,  ..., -0.9922, -0.9922, -0.9922],
         ...,
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]]], ),
 tensor(1.),
 tensor(21.5000))

In [ ]:
import random

train_size = len(ds)
valid_size = int(0.2 * train_size)

all_indices = list(range(train_size))
valid_ixs = random.sample(all_indices, valid_size)
train_ixs = list(set(all_indices) - set(valid_ixs))

train_ds = Subset(ds, train_ixs)
valid_ds = Subset(ds, valid_ixs)
test_ds  = XrayDataset('/content/test_images', transform=transforms)  # we don't have the labels, the goal is to predict them

train_ds[0] # notice we're now getting a normalized tensor

(Image([[[-1.0000, -1.0000, -1.0000,  ..., -0.7725, -0.7725, -0.7725],
         [-1.0000, -1.0000, -1.0000,  ..., -0.9216, -0.9216, -0.9216],
         [-1.0000, -1.0000, -1.0000,  ..., -0.9922, -0.9922, -0.9922],
         ...,
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]]], ),
 tensor(1.),
 tensor(21.5000))

In [ ]:
# Create dataloaders
bs = 48
train_loader = DataLoader(train_ds, batch_size=bs,   shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=2*bs, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=2*bs, shuffle=False)

bx, by_c, by_r = next(iter(train_loader))
bx.shape, by_c.shape, by_r.shape

(torch.Size([48, 1, 224, 224]), torch.Size([48]), torch.Size([48]))

## Task 3: Model and training

In [ ]:
# train on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Here we are now using a simple CNN from the lectures:

In [ ]:
# simple Convolutional Neural Net (CNN)
class SimpleCNN(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()

        # simple CNN from the lecture notebook
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels=1,  out_channels=16, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,16,h,w]
            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,16,h,w]
            nn.MaxPool2d(kernel_size=2, stride=2),                                                          # [bs,16,h/2,w/2]
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,32,h/2,w/2]
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,32,h/2,w/2]
            nn.AdaptiveAvgPool2d((4,4)),                                                                    # [bs,32,4,4]
            nn.Flatten()                                                                                    # [bs,32*4*4]
        )

        # this head will be used for the classification task (sex 0 or 1)
        self.clasf_head = nn.Sequential(
            nn.Linear(32*4*4, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),  # we only have two classes, so we could have either 2 or 1 output
            nn.Sigmoid()
        )

        # this head will be used for the BMI regression task
        # note that although it has almost the same architecture as the classification head,
        # it is a separate module with separate parameters!
        self.regr_head = nn.Sequential(
            nn.Linear(32*4*4, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),  # we only have one regression output (predicted BMI)
        )

    def forward(self, x):
        out = self.backbone(x)
        out_clasf = self.clasf_head(out)
        out_regr = self.regr_head(out)
        return out_clasf, out_regr  # not we have *two* outputs here!

# Instantiate the model
model = SimpleCNN(hidden_size=64)
model.to(device)
model

SimpleCNN(
  (backbone): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): LeakyReLU(negative_slope=0.01)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): LeakyReLU(negative_slope=0.01)
    (7): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): LeakyReLU(negative_slope=0.01)
    (9): AdaptiveAvgPool2d(output_size=(4, 4))
    (10): Flatten(start_dim=1, end_dim=-1)
  )
  (clasf_head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (regr_head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out

In [ ]:
import torch.nn as nn
import torch.optim as optim

# loss and optimiser
criterion_clasf = nn.BCELoss()
criterion_regr = nn.L1Loss()

model = SimpleCNN(hidden_size=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def one_epoch(model, loader, criterion_clasf, criterion_regr, optimizer=None):
    device = next(model.parameters()).device

    if optimizer is None:
        model.eval()
    else:
        model.train()

    running_loss = 0.0
    total_predictions = 0
    correct_predictions = 0
    losses = []

    with torch.set_grad_enabled(optimizer is not None): # like torch.no_grad() but only if no optimizer given

        for images, labels_c, labels_r in loader:
            images, labels_c, labels_r = images.to(device), labels_c.to(device), labels_r.to(device)

            # Forward pass
            outputs = model(images)
            loss_clasf = criterion_clasf(outputs[0], labels_c.unsqueeze(1))
            loss_regr = criterion_regr(outputs[1], labels_r.unsqueeze(1))
            loss = loss_clasf*10 + loss_regr
            # print(f"classification loss: {loss_clasf.item():0.4f}, regression loss: {loss_regr.item():0.2f}, ratio: {(loss_regr/loss_clasf).item():0.2f}")

            # Backward and optimize
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_predictions += images.size(0)
            running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / total_predictions

    return epoch_loss

In [ ]:
# training loop
num_epochs = 30

train_losses, valid_losses = [], []

for epoch in range(num_epochs):

    train_loss = one_epoch(model, train_loader, criterion_clasf, criterion_regr, optimizer)
    valid_loss = one_epoch(model, valid_loader, criterion_clasf, criterion_regr)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}], Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}')

Epoch [01/30], Training Loss: 27.4476, Validation Loss: 12.6127
Epoch [02/30], Training Loss: 11.1902, Validation Loss: 10.5124
Epoch [03/30], Training Loss: 10.0117, Validation Loss: 9.8568
Epoch [04/30], Training Loss: 9.5138, Validation Loss: 9.7828
Epoch [05/30], Training Loss: 9.4056, Validation Loss: 9.4730
Epoch [06/30], Training Loss: 8.9882, Validation Loss: 9.1611
Epoch [07/30], Training Loss: 8.7878, Validation Loss: 9.3914
Epoch [08/30], Training Loss: 8.5926, Validation Loss: 9.0787
Epoch [09/30], Training Loss: 8.1203, Validation Loss: 8.8119
Epoch [10/30], Training Loss: 7.8954, Validation Loss: 7.4513
Epoch [11/30], Training Loss: 7.5474, Validation Loss: 7.3949
Epoch [12/30], Training Loss: 7.4060, Validation Loss: 7.0672
Epoch [13/30], Training Loss: 7.1913, Validation Loss: 7.7061
Epoch [14/30], Training Loss: 7.1533, Validation Loss: 6.7761
Epoch [15/30], Training Loss: 6.9753, Validation Loss: 6.9175
Epoch [16/30], Training Loss: 7.0427, Validation Loss: 6.7013
Epo

## Task 4: Predictions for the holdout (i.e. the one held on Kaggle) dataset

In [ ]:
import numpy as np

model.eval()  # Set the model to evaluation mode
hold_out_pred_sex, hold_out_pred_bmi = [], []

with torch.no_grad():
    for images, _, _ in test_loader: # test_loader does not have labels
        images = images.to(device)
        outputs = model(images)
        predicted_sex = (outputs[0] > 0.5).float().cpu().numpy() # Apply threshold and convert to numpy
        predicted_bmi = outputs[1].cpu().numpy()

        hold_out_pred_sex.extend(predicted_sex.flatten().tolist())
        hold_out_pred_bmi.extend(predicted_bmi.flatten().tolist())

# Get the filenames for the test set
hold_out_filenames = test_ds.img_files

# Create a DataFrame for submission
submission_df = pd.DataFrame({'filename': hold_out_filenames, 'sex': hold_out_pred_sex, 'bmi': hold_out_pred_bmi})

# Kaggle submission requires the header and index=False
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print(submission_df.head())

Submission file created successfully!
        filename  sex        bmi
0  8152_4582.jpg  1.0  28.009331
1  1651_2690.jpg  0.0  24.298689
2  9247_2445.jpg  1.0  21.787468
3  9327_7908.jpg  1.0  22.239920
4  6156_4367.jpg  1.0  22.420979


## Task 5: Submitting to Kaggle

You can submit your predications via the Kaggle website or using the API. Either way, don’t forget to **add a brief description of the submission** before you upload. See how the submission stands on the leaderboard.

In [ ]:
!kaggle competitions submit -c bucv26 -f submission.csv -m 'YOUR DESCRIPTION!'

100% 4.78k/4.78k [00:00<00:00, 13.4kB/s]
Successfully submitted to BUCV26